## Bronze → Silver: Data Cleaning & Transformation

Reads 12 months of raw NYC TLC taxi trip data from bronze, resolves schema drift across files, applies data quality rules, and writes the cleaned result as a Delta table in silver.

**Flow:**
1. Read each month's file with its own native schema, explicitly casting to a consistent target schema (handles inconsistent types/casing across TLC's monthly files).
2. Union all 12 months into a single DataFrame.
3. Apply data quality rules (remove invalid passenger counts, distances, fares; fix placeholder rate codes).
4. Write the cleaned result as a partitioned Delta table.

**Note:** An earlier attempt using `mergeSchema` and a single unified read failed due to genuine schema drift (differing types and column casing across months) — resolved by reading and casting each month independently before unioning.

### Step 1: Read Each Month with Native Schema, Cast to Target Schema

Handles TLC's inconsistent column types/casing across months by reading each file natively, then explicitly casting each column to our target types and filling missing columns with typed nulls.

In [0]:
from pyspark.sql.types import *
from pyspark.sql.functions import col, lit

# Target schema — the final types we want, regardless of what each file has natively
target_types = {
    "VendorID": IntegerType(),
    "passenger_count": DoubleType(),
    "trip_distance": DoubleType(),
    "RatecodeID": DoubleType(),
    "store_and_fwd_flag": StringType(),
    "PULocationID": IntegerType(),
    "DOLocationID": IntegerType(),
    "payment_type": LongType(),
    "fare_amount": DoubleType(),
    "extra": DoubleType(),
    "mta_tax": DoubleType(),
    "tip_amount": DoubleType(),
    "tolls_amount": DoubleType(),
    "improvement_surcharge": DoubleType(),
    "total_amount": DoubleType(),
    "congestion_surcharge": DoubleType(),
    "airport_fee": DoubleType(),
}

spark.conf.set("spark.sql.caseSensitive", "false")

months = ["01","02","03","04","05","06","07","08","09","10","11","12"]
df_list = []
storage_account = "stnyctaxiraj01"
for m in months:
    path = f"abfss://bronze@{storage_account}.dfs.core.windows.net/taxi/year=2023/month={m}/"
    df_month = spark.read.parquet(path)  # let Spark infer this file's OWN native schema

    # Normalize column name casing (handles airport_fee vs Airport_fee)
    for c in df_month.columns:
        df_month = df_month.withColumnRenamed(c, c)  # no-op here since caseSensitive=false handles matching

    # Explicitly cast each target column to our desired type
    select_exprs = []
    for colname, dtype in target_types.items():
        matching_col = [c for c in df_month.columns if c.lower() == colname.lower()]
        if matching_col:
            select_exprs.append(col(matching_col[0]).cast(dtype).alias(colname))
        else:
            select_exprs.append(lit(None).cast(dtype).alias(colname))  # column missing in this month, fill null

    # Keep datetime columns as-is (already consistent) + add year/month labels
    df_month_clean = df_month.select(
        col("tpep_pickup_datetime"),
        col("tpep_dropoff_datetime"),
        *select_exprs
    ).withColumn("year", lit(2023)).withColumn("month", lit(int(m)))

    df_list.append(df_month_clean)

# Union all 12 months together
df_bronze = df_list[0]
for df in df_list[1:]:
    df_bronze = df_bronze.unionByName(df)

print(f"Total rows: {df_bronze.count()}")
df_bronze.printSchema()

df_bronze.select(
    "passenger_count", "trip_distance", "fare_amount", 
    "tip_amount", "total_amount", "RatecodeID"
).summary("count", "min", "max", "mean").show()

### Step 2: Apply Data Quality Rules

- Remove rows with null/zero passenger count
- Remove rows with zero or unrealistic (>100 mile) trip distance
- Remove rows with zero or unrealistic (>$500) fare amount
- Remove negative tip amounts
- Convert RatecodeID = 99 (TLC's "unknown" placeholder) to null

In [0]:
from pyspark.sql.functions import col, when

# Start from our unified, correctly-typed bronze DataFrame
df_silver = df_bronze

# Rule 1 & 2: passenger_count must be non-null and > 0
df_silver = df_silver.filter(
    (col("passenger_count").isNotNull()) & (col("passenger_count") > 0)
)

# Rule 2 & 3: trip_distance must be > 0 and <= 100 miles
df_silver = df_silver.filter(
    (col("trip_distance") > 0) & (col("trip_distance") <= 100)
)

# Rule 4 & 5: fare_amount must be > 0 and <= $500
df_silver = df_silver.filter(
    (col("fare_amount") > 0) & (col("fare_amount") <= 500)
)

# Rule 7: tip_amount must not be negative (0 is fine — many trips have no tip)
df_silver = df_silver.filter(col("tip_amount") >= 0)

# Rule 6: RatecodeID 99 is TLC's own "unknown" placeholder — convert to actual null
df_silver = df_silver.withColumn(
    "RatecodeID",
    when(col("RatecodeID") == 99, None).otherwise(col("RatecodeID"))
)

print(f"Rows before cleaning: {df_bronze.count()}")
print(f"Rows after cleaning: {df_silver.count()}")

### Step 3: Write Cleaned Data to Silver (Delta, partitioned by year/month)

In [0]:
silver_path = f"abfss://silver@{storage_account}.dfs.core.windows.net/taxi/"

df_silver.write \
    .format("delta") \
    .mode("overwrite") \
    .partitionBy("year", "month") \
    .save(silver_path)

print("Silver write complete.")

In [0]:
df_check = spark.read.format("delta").load(silver_path)
print(f"Silver row count: {df_check.count()}")
df_check.show(5)